# Required Submission Workflow: Non-Torch Model

This notebook is intentionally simple. The model is a tiny sklearn Ridge regressor. The same pattern works for XGBoost, LightGBM, CatBoost, sklearn, or any other non-Torch model: define reusable feature-building, reusable prediction, and log a reconstructable `model_submission/` bundle to MLflow. There is no required load helper; the required pieces are the feature builder, the prediction function, and the saved artifact/metadata.

## 1. Imports And Run Configuration

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
model_name = 'non_torch_required_submission_example'
horizon = 5
rebalance_frequency = 'weekly'  # choose: 'daily', 'weekly', or 'monthly'
target_col = f'forward_return_{horizon}d'
output_dir = repo_root / 'runs' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
print('Repo root:', repo_root)
print('Dataset:', dataset_name, spec.name)
print('Model name:', model_name)
print('Rebalance frequency:', rebalance_frequency)

## 2. Required Function: Build Model Features

This function is required. It must include every custom feature used by the model so Adam can rebuild the same inputs on another dataset.

In [ ]:
base_feature_names = [
    'momentum_20d',
    'vol_20d',
    'rsi_14',
    'price_to_sma_20d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
]

custom_feature_names = [
    'mom_vol_ratio',
]

feature_names = base_feature_names + custom_feature_names

def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    features = build_features(prices, feature_names=base_feature_names)
    features['mom_vol_ratio'] = features['momentum_20d'] / features['vol_20d'].replace(0.0, np.nan)
    return features.replace([np.inf, -np.inf], np.nan)

print('Feature names in model order:')
for feature in feature_names:
    print(' ', feature)

## 3. Train The Simple Non-Torch Model

This uses a sklearn `Pipeline` so scaling is saved inside the model artifact. For XGBoost/LightGBM/CatBoost, save their native model file and include preprocessing metadata in the manifest.

In [ ]:
prices = load_prices(dataset_name, repo_root=repo_root)
features = build_model_features(prices)
targets = make_forward_return_target(prices, horizon=horizon)

model_frame = (
    features
    .merge(targets[['date', 'ticker', target_col]], on=['date', 'ticker'], how='left')
    .dropna(subset=feature_names + [target_col])
    .reset_index(drop=True)
)

train = slice_split(model_frame, dataset_name, 'train', repo_root=repo_root)
test = slice_split(model_frame, dataset_name, 'test', repo_root=repo_root)

model = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0)),
    ]
)

model.fit(train[feature_names], train[target_col])
print('Trained Ridge pipeline')

## 4. Required Function: Save Submission Model Artifact

For this non-Torch example, the sklearn pipeline is saved with `joblib`. XGBoost/LightGBM/CatBoost users can replace this with their model library's native save method.

In [ ]:
model_path = output_dir / 'sklearn_ridge_pipeline.joblib'
joblib.dump(model, model_path)
print('Saved non-Torch model artifact:', model_path)

## 6. Required Function: Predict From Prices

This function takes a price frame, rebuilds features, uses the trained model object, and returns the standard prediction contract. In this notebook it uses the in-memory trained model. Later, Adam can manually load the saved artifact and pass that model object into this same function.

In [ ]:
def predict_from_prices(model, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    features = build_model_features(prices)
    if dates is not None:
        dates = pd.to_datetime(dates)
        features = features.loc[features['date'].isin(dates)].copy()
    if tickers is not None:
        tickers = [ticker.upper() for ticker in tickers]
        features = features.loc[features['ticker'].isin(tickers)].copy()

    scoring_frame = features.dropna(subset=feature_names).reset_index(drop=True)
    scores = model.predict(scoring_frame[feature_names])

    predictions = scoring_frame[['date', 'ticker']].copy()
    predictions['horizon'] = horizon
    predictions['expected_return'] = scores
    return predictions

def select_rebalance_dates(dates, frequency: str) -> pd.DatetimeIndex:
    date_index = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None).sort_values().unique())
    frequency = frequency.strip().lower()
    if frequency == 'daily':
        return date_index
    date_frame = pd.DataFrame({'date': date_index})
    if frequency == 'weekly':
        return pd.DatetimeIndex(date_frame.groupby(date_frame['date'].dt.to_period('W'))['date'].first().to_numpy())
    if frequency == 'monthly':
        return pd.DatetimeIndex(date_frame.groupby(date_frame['date'].dt.to_period('M'))['date'].first().to_numpy())
    raise ValueError("rebalance_frequency must be one of: 'daily', 'weekly', 'monthly'")

test_dates = pd.to_datetime(test['date'].unique())
rebalance_dates = select_rebalance_dates(test_dates, rebalance_frequency)
raw_predictions = predict_from_prices(model, prices, dates=rebalance_dates, tickers=spec.tickers)
predictions = validate_prediction_frame(raw_predictions, dataset_name=dataset_name, horizon=horizon, repo_root=repo_root)
print('Rebalance dates:', len(rebalance_dates), rebalance_dates.min(), 'to', rebalance_dates.max())
display(predictions.head())

## 7. Backtest The Submitted Predictions

In [ ]:
portfolio = weights_from_predictions_rank_long_only(predictions, dataset_name=dataset_name, strategy_name=model_name)
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=dataset_name, repo_root=repo_root)
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
artifact_paths = write_backtest_artifacts(result, output_dir)

print('Weights:', validated_weights.shape)
print('Metrics:')
for key, value in sorted(build_metrics(result).items()):
    print(f'  {key}: {value:.6f}')

## 8. Required MLflow Submission Logging

This logs normal research artifacts plus the reconstructable `model_submission/` bundle.

In [ ]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=f'{model_name}_submission',
    dataset_name=dataset_name,
    tags={
        'workflow': 'required_non_torch_submission',
        'model_family': 'sklearn',
        'prediction_horizon': str(horizon),
        'rebalance_frequency': rebalance_frequency,
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        'model_name': model_name,
        'dataset_name': dataset_name,
        'horizon': horizon,
        'feature_count': len(feature_names),
        'feature_list': ','.join(feature_names),
        'target': target_col,
        'rebalance_frequency': rebalance_frequency,
        'portfolio_builder': 'weights_from_predictions_rank_long_only',
    })

    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

    manifest = log_model_submission(
        {'sklearn_pipeline': model_path},
        model_name=model_name,
        model_family='sklearn',
        feature_names=feature_names,
        target=target_col,
        horizon=horizon,
        rebalance_frequency=rebalance_frequency,
        preprocessing={
            'scaler': 'inside_sklearn_pipeline',
        },
        model_config={
            'library': 'sklearn',
            'estimator': 'Ridge',
            'artifact_format': 'joblib',
            'rebalance_frequency': rebalance_frequency,
            'portfolio_builder': 'weights_from_predictions_rank_long_only',
            'required_functions': ['build_model_features', 'predict_from_prices'],
        },
        source_files=[repo_root / 'notebooks' / 'templates' / 'non_torch_required_submission_workflow.ipynb'],
        notes='Required non-Torch submission example with reusable feature and predict functions. No required load helper.',
    )

print(json.dumps(manifest, indent=2))